Script to filter the counts to keep only one sample per patient (the sample with the highest number of counts) for both AD and Healthy samples. This is done using the metadata information from the SRA run selector (PRJNA574438_SraRunTable.csv).

In [1]:
import pandas as pd

# === Step 1: Load data ===
meta = pd.read_csv("PRJNA574438_SraRunTable.csv")
counts = pd.read_csv("All-NCI-Aligned/All-NCI_Counts_v46_Clean.txt", sep="\t")

# === Step 2: Map SRR to Sample Name ===
srr_to_sample = dict(zip(meta["Run"], meta["Sample Name"]))

# Keep only SRR that are in both metadata and counts
valid_srrs = [c for c in counts.columns if c in srr_to_sample]

# Subset counts to only valid SRRs (plus Geneid)
counts_valid = counts[["Geneid"] + valid_srrs]

# === Step 3: Compute total counts per SRR ===
total_counts = counts_valid.drop(columns=["Geneid"]).sum(axis=0)

# Build dataframe with SRR, Sample Name, and total counts
sample_df = pd.DataFrame({
    "SRR": total_counts.index,
    "total_counts": total_counts.values
})
sample_df["Sample Name"] = sample_df["SRR"].map(srr_to_sample)

# === Step 4: Select best SRR per Sample Name ===
best_samples = sample_df.sort_values("total_counts", ascending=False).drop_duplicates("Sample Name")

# === Step 5: Subset counts table to keep only best SRRs ===
final_counts = counts_valid[["Geneid"] + best_samples["SRR"].tolist()]

# === Step 6: CPM normalisation ===
counts_only = final_counts.drop(columns=["Geneid"])
lib_sizes = counts_only.sum(axis=0)
cpm = (counts_only / lib_sizes) * 1e6
cpm.insert(0, "Geneid", final_counts["Geneid"])

# Save outputs
final_counts.to_csv("All-NCI_Counts_v46_Clean_UniquePatients.txt", sep="\t", index=False)
cpm.to_csv("All-NCI_Counts_v46_Clean_UniquePatients_CPM.txt", sep="\t", index=False)

print("Final matrix shape:", final_counts.shape)
print("Number of unique patients:", best_samples.shape[0])
print("CPM matrix shape:", cpm.shape)

Final matrix shape: (61471, 116)
Number of unique patients: 115
CPM matrix shape: (61471, 116)


In [2]:
import pandas as pd

# === Step 1: Load data ===
meta = pd.read_csv("PRJNA574438_SraRunTable.csv")
counts = pd.read_csv("All-AD-Aligned/All-AD_Counts_v46_Clean.txt", sep="\t")

# === Step 2: Map SRR to Sample Name ===
srr_to_sample = dict(zip(meta["Run"], meta["Sample Name"]))

# Keep only SRR that are in both metadata and counts
valid_srrs = [c for c in counts.columns if c in srr_to_sample]

# Subset counts to only valid SRRs (plus Geneid)
counts_valid = counts[["Geneid"] + valid_srrs]

# === Step 3: Compute total counts per SRR ===
total_counts = counts_valid.drop(columns=["Geneid"]).sum(axis=0)

# Build dataframe with SRR, Sample Name, and total counts
sample_df = pd.DataFrame({
    "SRR": total_counts.index,
    "total_counts": total_counts.values
})
sample_df["Sample Name"] = sample_df["SRR"].map(srr_to_sample)

# === Step 4: Select best SRR per Sample Name ===
best_samples = sample_df.sort_values("total_counts", ascending=False).drop_duplicates("Sample Name")

# === Step 5: Subset counts table to keep only best SRRs ===
final_counts = counts_valid[["Geneid"] + best_samples["SRR"].tolist()]

# === Step 6: CPM normalisation ===
counts_only = final_counts.drop(columns=["Geneid"])
lib_sizes = counts_only.sum(axis=0)
cpm = (counts_only / lib_sizes) * 1e6
cpm.insert(0, "Geneid", final_counts["Geneid"])

# Save outputs
final_counts.to_csv("All-AD_Counts_v46_Clean_UniquePatients.txt", sep="\t", index=False)
cpm.to_csv("All-AD_Counts_v46_Clean_UniquePatients_CPM.txt", sep="\t", index=False)

print("Final matrix shape:", final_counts.shape)
print("Number of unique patients:", best_samples.shape[0])
print("CPM matrix shape:", cpm.shape)

Final matrix shape: (61471, 128)
Number of unique patients: 127
CPM matrix shape: (61471, 128)
